# Input Bridge Comparison: Coffee vs Alcohol

This notebook tests which input variables better predict sleep efficiency.

Original input path:

coffee + exercise → sleep efficiency

Possible revised input path:

alcohol + exercise → sleep efficiency

The target variable is Sleep efficiency from the Sleep Efficiency dataset.

The models are observational and do not prove causality.

A model is only treated as usable if it beats the baseline mean model on held-out test RMSE.

In [2]:
# ------------------------------------------------------------
# Input bridge comparison: coffee vs alcohol
# ------------------------------------------------------------

from pathlib import Path
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Install packages if needed
# ------------------------------------------------------------

def install_if_missing(package_name, import_name=None):
    if import_name is None:
        import_name = package_name

    try:
        __import__(import_name)
    except ImportError:
        print(f"{package_name} missing. Installing...")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            package_name
        ])


install_if_missing("pandas")
install_if_missing("numpy")
install_if_missing("matplotlib")
install_if_missing("scikit-learn", "sklearn")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

DATA_DIR = PROJECT_DIR / "data"
GRAPH_DIR = PROJECT_DIR / "Graphs"

GRAPH_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:")
print(PROJECT_DIR)

# ------------------------------------------------------------
# Find Sleep_Efficiency.csv recursively
# ------------------------------------------------------------

sleep_efficiency_matches = list(PROJECT_DIR.rglob("Sleep_Efficiency.csv"))

if len(sleep_efficiency_matches) == 0:
    raise FileNotFoundError(
        "Could not find Sleep_Efficiency.csv in the project folder."
    )

sleep_efficiency_path = sleep_efficiency_matches[0]

print("\nUsing Sleep Efficiency dataset:")
print(sleep_efficiency_path)

sleep_raw = pd.read_csv(sleep_efficiency_path)

print("\nDataset shape:")
print(sleep_raw.shape)

print("\nColumns:")
print(list(sleep_raw.columns))

display(sleep_raw.head())

# ------------------------------------------------------------
# Prepare variables
# ------------------------------------------------------------

required_columns = [
    "Sleep efficiency",
    "Caffeine consumption",
    "Alcohol consumption",
    "Exercise frequency"
]

missing_columns = [
    column for column in required_columns
    if column not in sleep_raw.columns
]

if len(missing_columns) > 0:
    raise ValueError(f"Missing columns: {missing_columns}")

sleep_model_data = sleep_raw[required_columns].copy()

for column in required_columns:
    sleep_model_data[column] = pd.to_numeric(
        sleep_model_data[column],
        errors="coerce"
    )

sleep_model_data = sleep_model_data.dropna().copy()

print("\nRows available for fair comparison:")
print(len(sleep_model_data))

print("\nDescriptive statistics:")
display(sleep_model_data.describe().round(3))

print("\nCorrelations:")
display(sleep_model_data.corr().round(3))

# ------------------------------------------------------------
# Helper function: evaluate model and extract formula
# ------------------------------------------------------------

def evaluate_linear_formula(data, predictors, target, model_name):
    model_data = data[predictors + [target]].dropna().copy()

    X = model_data[predictors]
    y = model_data[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42
    )

    # --------------------------------------------------------
    # Baseline model
    # --------------------------------------------------------

    baseline = DummyRegressor(strategy="mean")
    baseline.fit(X_train, y_train)

    baseline_predictions = baseline.predict(X_test)

    baseline_r2 = r2_score(y_test, baseline_predictions)
    baseline_mae = mean_absolute_error(y_test, baseline_predictions)
    baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))

    # --------------------------------------------------------
    # Linear regression
    # --------------------------------------------------------

    linear_pipeline = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])

    linear_pipeline.fit(X_train, y_train)

    linear_train_predictions = linear_pipeline.predict(X_train)
    linear_test_predictions = linear_pipeline.predict(X_test)

    linear_train_r2 = r2_score(y_train, linear_train_predictions)
    linear_test_r2 = r2_score(y_test, linear_test_predictions)
    linear_mae = mean_absolute_error(y_test, linear_test_predictions)
    linear_rmse = np.sqrt(mean_squared_error(y_test, linear_test_predictions))

    # --------------------------------------------------------
    # Random forest comparison
    # --------------------------------------------------------

    random_forest = RandomForestRegressor(
        n_estimators=300,
        max_depth=4,
        min_samples_leaf=4,
        random_state=42
    )

    random_forest.fit(X_train, y_train)

    rf_train_predictions = random_forest.predict(X_train)
    rf_test_predictions = random_forest.predict(X_test)

    rf_train_r2 = r2_score(y_train, rf_train_predictions)
    rf_test_r2 = r2_score(y_test, rf_test_predictions)
    rf_mae = mean_absolute_error(y_test, rf_test_predictions)
    rf_rmse = np.sqrt(mean_squared_error(y_test, rf_test_predictions))

    # --------------------------------------------------------
    # Best model by RMSE
    # --------------------------------------------------------

    model_options = [
        ("baseline mean", baseline_rmse),
        ("linear regression", linear_rmse),
        ("random forest", rf_rmse)
    ]

    best_model, best_rmse = sorted(
        model_options,
        key=lambda item: item[1]
    )[0]

    beats_baseline = best_rmse < baseline_rmse

    # --------------------------------------------------------
    # Extract formula from linear model
    # --------------------------------------------------------

    scaler = linear_pipeline.named_steps["scaler"]
    linear_model = linear_pipeline.named_steps["model"]

    standardized_intercept = linear_model.intercept_
    standardized_coefficients = linear_model.coef_

    feature_means = scaler.mean_
    feature_sds = scaler.scale_

    raw_coefficients = standardized_coefficients / feature_sds

    raw_intercept = standardized_intercept - np.sum(
        standardized_coefficients * feature_means / feature_sds
    )

    standardized_parts = [
        f"{standardized_coefficients[i]:+.6f} * z({predictors[i]})"
        for i in range(len(predictors))
    ]

    raw_parts = [
        f"{raw_coefficients[i]:+.6f} * {predictors[i]}"
        for i in range(len(predictors))
    ]

    standardized_formula = (
        f"{target} = {standardized_intercept:.6f} "
        + " ".join(standardized_parts)
    )

    raw_formula = (
        f"{target} = {raw_intercept:.6f} "
        + " ".join(raw_parts)
    )

    result = {
        "model_name": model_name,
        "target": target,
        "predictors": ", ".join(predictors),
        "rows_used": len(model_data),
        "baseline_r2": round(baseline_r2, 4),
        "baseline_mae": round(baseline_mae, 4),
        "baseline_rmse": round(baseline_rmse, 4),
        "linear_train_r2": round(linear_train_r2, 4),
        "linear_test_r2": round(linear_test_r2, 4),
        "linear_mae": round(linear_mae, 4),
        "linear_rmse": round(linear_rmse, 4),
        "rf_train_r2": round(rf_train_r2, 4),
        "rf_test_r2": round(rf_test_r2, 4),
        "rf_mae": round(rf_mae, 4),
        "rf_rmse": round(rf_rmse, 4),
        "best_model": best_model,
        "beats_baseline": "Yes" if beats_baseline else "No",
        "decision": (
            "use with caution"
            if beats_baseline and best_model != "baseline mean"
            else "do not use"
        ),
        "standardized_formula": standardized_formula,
        "raw_formula": raw_formula
    }

    coefficient_rows = []

    coefficient_rows.append({
        "model_name": model_name,
        "term": "Intercept",
        "standardized_coefficient": standardized_intercept,
        "raw_coefficient": raw_intercept,
        "training_mean": "",
        "training_sd": ""
    })

    for i, predictor in enumerate(predictors):
        coefficient_rows.append({
            "model_name": model_name,
            "term": predictor,
            "standardized_coefficient": standardized_coefficients[i],
            "raw_coefficient": raw_coefficients[i],
            "training_mean": feature_means[i],
            "training_sd": feature_sds[i]
        })

    coefficient_table = pd.DataFrame(coefficient_rows)

    return result, coefficient_table

# ------------------------------------------------------------
# Compare input options
# ------------------------------------------------------------

target = "Sleep efficiency"

model_jobs = [
    {
        "model_name": "coffee + exercise → sleep efficiency",
        "predictors": ["Caffeine consumption", "Exercise frequency"]
    },
    {
        "model_name": "alcohol + exercise → sleep efficiency",
        "predictors": ["Alcohol consumption", "Exercise frequency"]
    },
    {
        "model_name": "coffee + alcohol + exercise → sleep efficiency",
        "predictors": ["Caffeine consumption", "Alcohol consumption", "Exercise frequency"]
    },
    {
        "model_name": "coffee only → sleep efficiency",
        "predictors": ["Caffeine consumption"]
    },
    {
        "model_name": "alcohol only → sleep efficiency",
        "predictors": ["Alcohol consumption"]
    },
    {
        "model_name": "exercise only → sleep efficiency",
        "predictors": ["Exercise frequency"]
    }
]

comparison_results = []
coefficient_tables = []

for job in model_jobs:
    result, coefficient_table = evaluate_linear_formula(
        data=sleep_model_data,
        predictors=job["predictors"],
        target=target,
        model_name=job["model_name"]
    )

    comparison_results.append(result)
    coefficient_tables.append(coefficient_table)

input_bridge_comparison_table = pd.DataFrame(comparison_results)
input_bridge_coefficient_table = pd.concat(coefficient_tables, ignore_index=True)

print("\nInput bridge comparison results:")
display(input_bridge_comparison_table)

print("\nCoefficient table:")
display(input_bridge_coefficient_table)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

comparison_path = PROJECT_DIR / "sleep_efficiency_input_bridge_comparison_results.csv"
coefficient_path = PROJECT_DIR / "sleep_efficiency_input_bridge_comparison_coefficients.csv"

input_bridge_comparison_table.to_csv(comparison_path, index=False)
input_bridge_coefficient_table.to_csv(coefficient_path, index=False)

print("\nSaved comparison results to:")
print(comparison_path)

print("\nSaved coefficients to:")
print(coefficient_path)

# ------------------------------------------------------------
# Plot RMSE comparison
# ------------------------------------------------------------

plot_data = input_bridge_comparison_table.copy()

plt.figure(figsize=(10, 6))
plt.barh(plot_data["model_name"], plot_data["linear_rmse"])
plt.xlabel("Linear model test RMSE")
plt.ylabel("Input model")
plt.title("Coffee vs Alcohol input bridge comparison")
plt.tight_layout()

graph_path = GRAPH_DIR / "coffee_vs_alcohol_sleep_efficiency_rmse.png"
plt.savefig(graph_path, dpi=200)
plt.close()

print("\nSaved RMSE graph to:")
print(graph_path)

# ------------------------------------------------------------
# Print best result
# ------------------------------------------------------------

best_linear_row = input_bridge_comparison_table.sort_values("linear_rmse").iloc[0]

print("\nBest linear formula by RMSE:")
print(best_linear_row["model_name"])
print(best_linear_row["raw_formula"])

print("\nBest model summary:")
print(f"Model: {best_linear_row['model_name']}")
print(f"Linear test R²: {best_linear_row['linear_test_r2']}")
print(f"Linear RMSE: {best_linear_row['linear_rmse']}")
print(f"Baseline RMSE: {best_linear_row['baseline_rmse']}")
print(f"Decision: {best_linear_row['decision']}")

Project folder:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis

Using Sleep Efficiency dataset:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\data\external_sleep_datasets\Sleep_Efficiency.csv

Dataset shape:
(452, 15)

Columns:
['ID', 'Age', 'Gender', 'Bedtime', 'Wakeup time', 'Sleep duration', 'Sleep efficiency', 'REM sleep percentage', 'Deep sleep percentage', 'Light sleep percentage', 'Awakenings', 'Caffeine consumption', 'Alcohol consumption', 'Smoking status', 'Exercise frequency']


,ID,Age,Gender,Bedtime,Wakeup time,Sleep duration,Sleep efficiency,REM sleep percentage,Deep sleep percentage,Light sleep percentage,Awakenings,Caffeine consumption,Alcohol consumption,Smoking status,Exercise frequency
0,1,65,Female,2021-03-06 01:00:00,2021-03-06 07:00:00,6.0,0.88,18,70,12,0.0,0.0,0.0,Yes,3.0
1,2,69,Male,2021-12-05 02:00:00,2021-12-05 09:00:00,7.0,0.66,19,28,53,3.0,0.0,3.0,Yes,3.0
2,3,40,Female,2021-05-25 21:30:00,2021-05-25 05:30:00,8.0,0.89,20,70,10,1.0,0.0,0.0,No,3.0
3,4,40,Female,2021-11-03 02:30:00,2021-11-03 08:30:00,6.0,0.51,23,25,52,3.0,50.0,5.0,Yes,1.0
4,5,57,Male,2021-03-13 01:00:00,2021-03-13 09:00:00,8.0,0.76,27,55,18,3.0,0.0,3.0,No,3.0



Rows available for fair comparison:
407

Descriptive statistics:


,Sleep efficiency,Caffeine consumption,Alcohol consumption,Exercise frequency
count,407.000,407.000,407.000,407.000
mean,0.790,23.034,1.162,1.779
std,0.135,28.978,1.608,1.447
min,0.500,0.000,0.000,0.000
25%,0.700,0.000,0.000,0.000
50%,0.820,25.000,0.000,2.000
75%,0.900,50.000,2.000,3.000
max,0.990,200.000,5.000,5.000



Correlations:


...


Input bridge comparison results:


,model_name,target,predictors,rows_used,baseline_r2,baseline_mae,baseline_rmse,linear_train_r2,linear_test_r2,linear_mae,linear_rmse,rf_train_r2,rf_test_r2,rf_mae,rf_rmse,best_model,beats_baseline,decision,standardized_formula,raw_formula
0,coffee + exercise → sleep efficiency,Sleep efficiency,"Caffeine consumption, Exercise frequency",407,-0.0005,0.1154,0.1374,0.0651,0.0981,0.1078,0.1305,0.1250,0.1003,0.1065,0.1303,random forest,Yes,use with caution,Sleep efficiency = 0.790492 +0.004844 * z(Caff...,Sleep efficiency = 0.743037 +0.000169 * Caffei...
1,alcohol + exercise → sleep efficiency,Sleep efficiency,"Alcohol consumption, Exercise frequency",407,-0.0005,0.1154,0.1374,0.2117,0.2339,0.0943,0.1203,0.4165,0.2511,0.0871,0.1189,random forest,Yes,use with caution,Sleep efficiency = 0.790492 -0.051567 * z(Alco...,Sleep efficiency = 0.785374 -0.031671 * Alcoho...
2,coffee + alcohol + exercise → sleep efficiency,Sleep efficiency,"Caffeine consumption, Alcohol consumption, Exe...",407,-0.0005,0.1154,0.1374,0.2117,0.2343,0.0943,0.1202,0.4165,0.2804,0.0853,0.1165,random forest,Yes,use with caution,Sleep efficiency = 0.790492 +0.000193 * z(Caff...,Sleep efficiency = 0.785183 +0.000007 * Caffei...
3,coffee only → sleep efficiency,Sleep efficiency,Caffeine consumption,407,-0.0005,0.1154,0.1374,0.0003,0.0051,0.1151,0.1370,0.0195,0.0167,0.1143,0.1362,random forest,Yes,use with caution,Sleep efficiency = 0.790492 +0.002137 * z(Caff...,Sleep efficiency = 0.788734 +0.000074 * Caffei...
4,alcohol only → sleep efficiency,Sleep efficiency,Alcohol consumption,407,-0.0005,0.1154,0.1374,0.1466,0.1415,0.1016,0.1273,0.2096,0.1575,0.0982,0.1261,random forest,Yes,use with caution,Sleep efficiency = 0.790492 -0.051332 * z(Alco...,Sleep efficiency = 0.828634 -0.031527 * Alcoho...
5,exercise only → sleep efficiency,Sleep efficiency,Exercise frequency,407,-0.0005,0.1154,0.1374,0.0638,0.0848,0.1087,0.1314,0.0906,0.0716,0.1093,0.1324,linear regression,Yes,use with caution,Sleep efficiency = 0.790492 +0.033851 * z(Exer...,Sleep efficiency = 0.747507 +0.023495 * Exerci...



Coefficient table:


,model_name,term,standardized_coefficient,raw_coefficient,training_mean,training_sd
0,coffee + exercise → sleep efficiency,Intercept,0.790492,0.743037,,
1,coffee + exercise → sleep efficiency,Caffeine consumption,0.004844,0.000169,23.606557,28.703275
2,coffee + exercise → sleep efficiency,Exercise frequency,0.034234,0.023761,1.829508,1.440781
3,alcohol + exercise → sleep efficiency,Intercept,0.790492,0.785374,,
4,alcohol + exercise → sleep efficiency,Alcohol consumption,-0.051567,-0.031671,1.209836,1.628205
5,alcohol + exercise → sleep efficiency,Exercise frequency,0.034205,0.023741,1.829508,1.440781
6,coffee + alcohol + exercise → sleep efficiency,Intercept,0.790492,0.785183,,
7,coffee + alcohol + exercise → sleep efficiency,Caffeine consumption,0.000193,0.000007,23.606557,28.703275
8,coffee + alcohol + exercise → sleep efficiency,Alcohol consumption,-0.051549,-0.031660,1.209836,1.628205
9,coffee + alcohol + exercise → sleep efficiency,Exercise frequency,0.034220,0.023751,1.829508,1.440781



Saved comparison results to:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\sleep_efficiency_input_bridge_comparison_results.csv

Saved coefficients to:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\sleep_efficiency_input_bridge_comparison_coefficients.csv

Saved RMSE graph to:
D:\Documentos\Sleep datasets\sleep_installation_data_analysis\Graphs\coffee_vs_alcohol_sleep_efficiency_rmse.png

Best linear formula by RMSE:
coffee + alcohol + exercise → sleep efficiency
Sleep efficiency = 0.785183 +0.000007 * Caffeine consumption -0.031660 * Alcohol consumption +0.023751 * Exercise frequency

Best model summary:
Model: coffee + alcohol + exercise → sleep efficiency
Linear test R²: 0.2343
Linear RMSE: 0.1202
Baseline RMSE: 0.1374
Decision: use with caution


In [3]:
# ------------------------------------------------------------
# Print clean formulas for coffee/alcohol/exercise models
# ------------------------------------------------------------

formula_columns = [
    "model_name",
    "baseline_rmse",
    "linear_test_r2",
    "linear_rmse",
    "rf_test_r2",
    "rf_rmse",
    "best_model",
    "beats_baseline",
    "decision",
    "raw_formula",
    "standardized_formula"
]

clear_formula_table = input_bridge_comparison_table[formula_columns].copy()

display(clear_formula_table)

print("CLEAR INPUT-BRIDGE FORMULAS")
print("=" * 80)

for _, row in clear_formula_table.iterrows():
    print("\n" + row["model_name"])
    print("-" * 80)
    print(f"Baseline RMSE: {row['baseline_rmse']}")
    print(f"Linear test R²: {row['linear_test_r2']}")
    print(f"Linear RMSE: {row['linear_rmse']}")
    print(f"Random forest test R²: {row['rf_test_r2']}")
    print(f"Random forest RMSE: {row['rf_rmse']}")
    print(f"Best model: {row['best_model']}")
    print(f"Beats baseline: {row['beats_baseline']}")
    print(f"Decision: {row['decision']}")
    print("\nRaw formula:")
    print(row["raw_formula"])
    print("\nStandardized formula:")
    print(row["standardized_formula"])

,model_name,baseline_rmse,linear_test_r2,linear_rmse,rf_test_r2,rf_rmse,best_model,beats_baseline,decision,raw_formula,standardized_formula
0,coffee + exercise → sleep efficiency,0.1374,0.0981,0.1305,0.1003,0.1303,random forest,Yes,use with caution,Sleep efficiency = 0.743037 +0.000169 * Caffei...,Sleep efficiency = 0.790492 +0.004844 * z(Caff...
1,alcohol + exercise → sleep efficiency,0.1374,0.2339,0.1203,0.2511,0.1189,random forest,Yes,use with caution,Sleep efficiency = 0.785374 -0.031671 * Alcoho...,Sleep efficiency = 0.790492 -0.051567 * z(Alco...
2,coffee + alcohol + exercise → sleep efficiency,0.1374,0.2343,0.1202,0.2804,0.1165,random forest,Yes,use with caution,Sleep efficiency = 0.785183 +0.000007 * Caffei...,Sleep efficiency = 0.790492 +0.000193 * z(Caff...
3,coffee only → sleep efficiency,0.1374,0.0051,0.1370,0.0167,0.1362,random forest,Yes,use with caution,Sleep efficiency = 0.788734 +0.000074 * Caffei...,Sleep efficiency = 0.790492 +0.002137 * z(Caff...
4,alcohol only → sleep efficiency,0.1374,0.1415,0.1273,0.1575,0.1261,random forest,Yes,use with caution,Sleep efficiency = 0.828634 -0.031527 * Alcoho...,Sleep efficiency = 0.790492 -0.051332 * z(Alco...
5,exercise only → sleep efficiency,0.1374,0.0848,0.1314,0.0716,0.1324,linear regression,Yes,use with caution,Sleep efficiency = 0.747507 +0.023495 * Exerci...,Sleep efficiency = 0.790492 +0.033851 * z(Exer...


CLEAR INPUT-BRIDGE FORMULAS

coffee + exercise → sleep efficiency
--------------------------------------------------------------------------------
Baseline RMSE: 0.1374
Linear test R²: 0.0981
Linear RMSE: 0.1305
Random forest test R²: 0.1003
Random forest RMSE: 0.1303
Best model: random forest
Beats baseline: Yes
Decision: use with caution

Raw formula:
Sleep efficiency = 0.743037 +0.000169 * Caffeine consumption +0.023761 * Exercise frequency

Standardized formula:
Sleep efficiency = 0.790492 +0.004844 * z(Caffeine consumption) +0.034234 * z(Exercise frequency)

alcohol + exercise → sleep efficiency
--------------------------------------------------------------------------------
Baseline RMSE: 0.1374
Linear test R²: 0.2339
Linear RMSE: 0.1203
Random forest test R²: 0.2511
Random forest RMSE: 0.1189
Best model: random forest
Beats baseline: Yes
Decision: use with caution

Raw formula:
Sleep efficiency = 0.785374 -0.031671 * Alcohol consumption +0.023741 * Exercise frequency

Standardiz

In [4]:
# ------------------------------------------------------------
# Print only the most important alcohol-related formulas
# ------------------------------------------------------------

models_to_print = [
    "alcohol + exercise → sleep efficiency",
    "coffee + alcohol + exercise → sleep efficiency",
    "alcohol only → sleep efficiency"
]

selected_formula_table = input_bridge_comparison_table[
    input_bridge_comparison_table["model_name"].isin(models_to_print)
].copy()

display(selected_formula_table[
    [
        "model_name",
        "baseline_rmse",
        "linear_test_r2",
        "linear_rmse",
        "rf_test_r2",
        "rf_rmse",
        "best_model",
        "beats_baseline",
        "decision",
        "raw_formula",
        "standardized_formula"
    ]
])

print("IMPORTANT ALCOHOL FORMULAS")
print("=" * 80)

for _, row in selected_formula_table.iterrows():
    print("\n" + row["model_name"])
    print("-" * 80)
    print(f"Baseline RMSE: {row['baseline_rmse']}")
    print(f"Linear test R²: {row['linear_test_r2']}")
    print(f"Linear RMSE: {row['linear_rmse']}")
    print(f"Random forest test R²: {row['rf_test_r2']}")
    print(f"Random forest RMSE: {row['rf_rmse']}")
    print(f"Best model: {row['best_model']}")
    print(f"Beats baseline: {row['beats_baseline']}")
    print(f"Decision: {row['decision']}")

    print("\nRaw formula:")
    print(row["raw_formula"])

    print("\nStandardized formula:")
    print(row["standardized_formula"])

,model_name,baseline_rmse,linear_test_r2,linear_rmse,rf_test_r2,rf_rmse,best_model,beats_baseline,decision,raw_formula,standardized_formula
1,alcohol + exercise → sleep efficiency,0.1374,0.2339,0.1203,0.2511,0.1189,random forest,Yes,use with caution,Sleep efficiency = 0.785374 -0.031671 * Alcoho...,Sleep efficiency = 0.790492 -0.051567 * z(Alco...
2,coffee + alcohol + exercise → sleep efficiency,0.1374,0.2343,0.1202,0.2804,0.1165,random forest,Yes,use with caution,Sleep efficiency = 0.785183 +0.000007 * Caffei...,Sleep efficiency = 0.790492 +0.000193 * z(Caff...
4,alcohol only → sleep efficiency,0.1374,0.1415,0.1273,0.1575,0.1261,random forest,Yes,use with caution,Sleep efficiency = 0.828634 -0.031527 * Alcoho...,Sleep efficiency = 0.790492 -0.051332 * z(Alco...


IMPORTANT ALCOHOL FORMULAS

alcohol + exercise → sleep efficiency
--------------------------------------------------------------------------------
Baseline RMSE: 0.1374
Linear test R²: 0.2339
Linear RMSE: 0.1203
Random forest test R²: 0.2511
Random forest RMSE: 0.1189
Best model: random forest
Beats baseline: Yes
Decision: use with caution

Raw formula:
Sleep efficiency = 0.785374 -0.031671 * Alcohol consumption +0.023741 * Exercise frequency

Standardized formula:
Sleep efficiency = 0.790492 -0.051567 * z(Alcohol consumption) +0.034205 * z(Exercise frequency)

coffee + alcohol + exercise → sleep efficiency
--------------------------------------------------------------------------------
Baseline RMSE: 0.1374
Linear test R²: 0.2343
Linear RMSE: 0.1202
Random forest test R²: 0.2804
Random forest RMSE: 0.1165
Best model: random forest
Beats baseline: Yes
Decision: use with caution

Raw formula:
Sleep efficiency = 0.785183 +0.000007 * Caffeine consumption -0.031660 * Alcohol consumption +0